# DINGO: direct function calls

Use the **grism** kernel. Run each cell when you want that step to happen. Every setting is an argument to a function.

`run_a` and `run_b` below are **two independent configurations of the same source**. Each has its own output folder. You can keep both in memory, run them back to back, and inspect either whenever you want. Change the ID in a `prepare_source` call to fit a different source.

The examples compare starting velocities of 400 and 450 km/s with an imaging inclination prior of width 5°. These starting values are examples, not measured velocities. The model remains the joint R+C rotation-only fit, sampling deprojected $V_\infty$ and inclination.

[Function reference and options](docs/workflow.md).

## Imports and file locations

`DATA_DIR/ID<id>/` contains `image.fits`, `R.fits`, `C.fits` and `config_kinematics.yaml`. If the configuration is elsewhere, pass `config_path=...` to `prepare_source`.

PySersic uses calibrated F150W science/error imaging and a matching PSF. Its local survey defaults are used below; pass `science=...`, `error=...`, `psf=...` to `fit_morphology` to select your own files.

In [ ]:
from pathlib import Path
from IPython.display import display
import matplotlib.pyplot as plt
from dingo import workflow as wf

DINGO_DIR = Path(wf.__file__).resolve().parents[1]
DATA_DIR = DINGO_DIR / "data"  # Change the input location here.
WORKING_DIR = DINGO_DIR.parent / "field_kinematics" / "edr_kinematics" / "PaA" / "dingo_runs"
print("Save location:", WORKING_DIR)

## Setup A — create the configuration

`prepare_source` records settings and checks the three inputs; it does not fit. Give each experiment a different `run_name`. Repeating an unchanged call reopens its run. Set `overwrite=True` in that call to reset that particular kinematic run.

Omit `start_velocity_kms` to use the velocity of your chosen search candidate instead.

In [ ]:
run_a = wf.prepare_source(
    15665,
    data_dir=DATA_DIR,
    working_dir=WORKING_DIR,
    run_name="15665_imaging_s5_v400",
    inclination_mode="imaging_prior",
    inclination_sigma_deg=5.0,
    start_velocity_kms=400.0,
    device="cuda",
    overwrite=False,
)
_ = wf.show_inputs(run_a)
print(run_a.directory)

### Fit or reuse PySersic

This fits one Sérsic component with a constant sky and displays the image, residuals and native corner plot. A matching fit is reused from the source's `morph_fits` directory. Its readable report is `pysersic_<ID>_info.cat`; it also saves `parameters.txt`, posterior samples and plots. Inclination is derived and displayed for $q_0=0$ and $q_0=0.2$.

`warmup` and `draws` here control PySersic, separately from the later kinematic MCMC.

In [ ]:
imaging_a = wf.fit_morphology(run_a, warmup=1000, draws=2000)
print(imaging_a.path)

### Search and inspect candidates

This searches for possible starting geometries and displays their velocities and residuals. It does not start MCMC.

In [ ]:
candidates_a = wf.find_candidates(run_a)
_ = wf.show_candidates(run_a)

### Choose the starting solution

Enter the original candidate ID from the table. The chosen candidate supplies the geometry; the explicit `start_velocity_kms=400` above supplies the starting velocity. This cell previews that exact start.

In [ ]:
candidate_a = int(input("Setup A: candidate ID selected from the residuals: "))
start_a = wf.prepare_start(run_a, candidate_id=candidate_a)
figure = start_a.figure()
display(figure)
plt.close(figure)

### Run the kinematic MCMC

**`fit_source` is the function that starts MCMC.** Change its arguments here. `draws` is the total retained target per walker, after warmup. Increase only `draws` to extend this same run; choose a new run for changed starts, priors, warmup, walkers or seed.

The GPU evaluates the PyTorch likelihood; emcee proposes ensemble moves. Progress bars and checkpoints are automatic. These 1,000 retained steps are a pilot, not a guarantee of converged uncertainties.

In [ ]:
posterior_a = wf.fit_source(
    run_a,
    warmup=1000,
    draws=1000,
    walkers=64,
    seed=42,
)

### Display and save A

Residuals show the highest-posterior saved draw; corner plots summarize pooled medians and 16th–84th percentiles. Inspect all walkers. Default exported radial profiles use illustrative radii; reporting masses requires justified observed radii.

In [ ]:
_ = wf.show_results(run_a)
saved_a = wf.save_results(run_a)
print(saved_a)

## Setup B — another configuration of the same source

Use another variable and another `run_name`. This example changes the starting velocity to 450 km/s. Setup A remains available as `run_a`.

To try a fixed inclination, change the arguments in the next call to `inclination_mode="fixed", inclination_deg=30.0` and choose a descriptive new run name. For a Gaussian with your chosen mean, use `inclination_mode="gaussian_prior", inclination_deg=30.0, inclination_sigma_deg=5.0`.

In [ ]:
run_b = wf.prepare_source(
    15665,
    data_dir=DATA_DIR,
    working_dir=WORKING_DIR,
    run_name="15665_imaging_s5_v450",
    inclination_mode="imaging_prior",
    inclination_sigma_deg=5.0,
    start_velocity_kms=450.0,
    device="cuda",
    overwrite=False,
)
print(run_b.directory)

### Reuse imaging, then search B

The matching source-level PySersic fit is reused. B performs its own candidate search, so it also works as a separate experiment without needing A's search results. For fixed inclination the exploratory search still varies inclination; `prepare_start` applies the exact fixed angle before MCMC.

In [ ]:
imaging_b = wf.fit_morphology(run_b, warmup=1000, draws=2000)
candidates_b = wf.find_candidates(run_b)
_ = wf.show_candidates(run_b)

### Choose and preview B

In [ ]:
candidate_b = int(input("Setup B: candidate ID selected from the residuals: "))
start_b = wf.prepare_start(run_b, candidate_id=candidate_b)
figure = start_b.figure()
display(figure)
plt.close(figure)

### Run B, then display and save

In [ ]:
posterior_b = wf.fit_source(
    run_b,
    warmup=1000,
    draws=1000,
    walkers=64,
    seed=42,
)

In [ ]:
_ = wf.show_results(run_b)
saved_b = wf.save_results(run_b)
print(saved_b)

## Use either configuration directly

These are ordinary Python objects. Run just the call you need in a new cell:

```python
wf.show_results(run_a)                  # Inspect A again.
wf.show_results(run_b)                  # Inspect B again.
wf.fit_source(run_a, warmup=1000, draws=8000, walkers=64, seed=42)  # Extend A.
```

For a third setup, copy a `prepare_source` call into a new cell, assign it to `run_c`, change its arguments and run name, then call the same separate functions with `run_c`.

After a kernel restart, execute the imports cell and reopen a saved run without fitting:

```python
run_a = wf.load_run(WORKING_DIR / "15665/15665_imaging_s5_v400")
wf.show_results(run_a)
```

To intentionally refit morphology, use `wf.fit_morphology(new_run, refit=True, ...)` on a new or explicitly reset kinematic run. This creates another morphology revision; existing experiments retain the revision they used.